Baseado no Gemini https://gemini.google.com/app/88365b3c7056ebd5

1. Preparação do Ambiente
Antes de abrir o Python, você precisa do "motor" que rodará o modelo:

Instale o Ollama: Baixe em ollama.com. Após instalar, abra o terminal do seu computador e digite:
ollama run llama3 (isso baixará o modelo de 8 bilhões de parâmetros, que é ótimo para português).

No VS Code: Instale a extensão "Jupyter" e a extensão "Python".

Bibliotecas Necessárias: No seu terminal do VS Code, instale:

pip install langchain_community langchain_core pandas



2. Estrutura do Código no Jupyter Notebook
Crie um arquivo .ipynb e siga estas etapas:

Passo A: Conexão com o Modelo Local
Nesta parte, configuramos o Python para "conversar" com o Ollama que está rodando no seu PC.

In [ ]:
from langchain_community.llms import Ollama

# Inicializa o Llama 3 localmente
llm = Ollama(model="dolphin-llama3:latest")

# Teste simples
# print(llm.invoke("Olá, você entende textos sobre segurança pública no Brasil?"))

: 

Passo B: Implementação Zero-Shot
No Zero-Shot, o modelo classifica o texto sem exemplos prévios, apenas com base na instrução (Prompt).

In [17]:
def classificar_zero_shot(texto):
    prompt = f"""
    Analise o seguinte texto jornalístico em português e classifique o comportamento da narrativa em relação à atuação policial.
    
    Categorias:
    - Pró-polícia (comportamento violento): Quando o texto exalta o uso da força, justifica violência sem questionamento ou foca apenas no sucesso da repressão.
    - Neutro: Relato factual, sem juízo de valor explícito sobre a conduta policial.
    - Crítico: Quando o texto questiona a legalidade, a eficiência ou a violência da ação policial.

    Texto: {texto}
    
    Responda APENAS com a categoria.
    """
    return llm.invoke(prompt).strip()

# Exemplo com o título da sua notícia
noticia_exemplo = "Jequié atinge marca de 30 dias sem crimes violentos após troca de comando na PM e operações da PC"
print(f"Resultado Zero-Shot: {classificar_zero_shot(noticia_exemplo)}")

Resultado Zero-Shot: Neutro


3. Implementação Few-Shot (Modelo Alternativo)
O Few-Shot é mais preciso para temas complexos porque você "ensina" o modelo com exemplos antes de pedir a classificação.

In [19]:
def classificar_few_shot(texto):
    prompt = f"""
    Você é um especialista em análise de mídia e ciência política. Classifique os textos sobre comportamento policial.

    Exemplo 1: "Ação competente da PM elimina suspeitos em confronto." -> Classificação: Pró-polícia (comportamento violento)
    Exemplo 2: "A Polícia Civil cumpriu três mandados de busca na manhã de hoje." -> Classificação: Neutro
    Exemplo 3: "Moradores denunciam abusos e tiros disparados por policiais durante incursão na favela." -> Classificação: Crítico

    Agora classifique este:
    Texto: {texto}
    Classificação:
    """
    return llm.invoke(prompt).strip()

In [21]:
noticia_exemplo = "Jequié atinge marca de 30 dias sem crimes violentos após troca de comando na PM e operações da PC"
print(f"Resultado Few-Shot: {classificar_few_shot(noticia_exemplo)}")

Resultado Few-Shot: Positivo (mudança na direção correta)


4. Fluxo Sugerido para Pesquisa Acadêmica
Para um projeto de doutorado, a consistência é fundamental. Aqui está uma tabela de como organizar seus dados para análise posterior:

Etapa,Ferramenta/Ação
Coleta,Use bibliotecas como BeautifulSoup ou Newspaper3k para extrair o texto do link.
Processamento,"Limpe o texto (remover HTML, excesso de espaços)."
Análise,Rode o modelo e salve o resultado em um CSV ou Excel usando Pandas.
Validação,Selecione 10% da amostra e classifique manualmente para calcular a acurácia do Llama.

No Python, a biblioteca newspaper3k é a padrão ouro para jornalismo: ela identifica automaticamente o que é título, o que é o corpo do texto e até a data de publicação, ignorando anúncios e menus do site.

Aqui está o código que você pode adicionar ao seu Jupyter Notebook para extrair o conteúdo do link de Jequié (e de outros que virão):

1. Instale a biblioteca de extração
No terminal do VS Code:

pip install newspaper3k


2. Código para extrair a notícia
Crie uma nova célula no seu Notebook e cole o seguinte:

In [3]:
from newspaper import Article

def extrair_conteudo_noticia(url):
    try:
        # Configura a extração para o link fornecido
        artigo = Article(url, language='pt')
        
        # Faz o download e o processamento do HTML
        artigo.download()
        artigo.parse()
        
        # Organiza os dados em um dicionário
        dados = {
            "titulo": artigo.title,
            "texto": artigo.text,
            "data": artigo.publish_date,
            "url": url
        }
        
        return dados
    except Exception as e:
        return f"Erro ao acessar a URL: {e}"

# Teste com o link de Jequié
url_bahia = "https://www.jequienews.com/noticia/jequie-atinge-marca-de-30-dias-sem-crimes-violentos-apos-troca-de-comando-na-pm-e-operacoes-da-pc"
noticia_extraida = extrair_conteudo_noticia(url_bahia)

if isinstance(noticia_extraida, dict):
    print(f"Título: {noticia_extraida['titulo']}")
    print("-" * 30)
    print(f"Texto (primeiros 300 caracteres): {noticia_extraida['texto'][:300]}...")
else:
    print(noticia_extraida)

Título: JEQUIÉ ATINGE MARCA DE 30 DIAS SEM CRIMES VIOLENTOS APÓS TROCA DE COMANDO NA PM E OPERAÇÕES DA PC
------------------------------
Texto (primeiros 300 caracteres): As mudanças recentes no policiamento ostensivo e investigativo no município de Jequié, como mudança no comando do 19º Batalhão de Polícia Militar e designação de um delegado para investigação dos crimes violentos letais intencionais (CVLIs), em especial Homicídios e Tentativas de homicídios dolosos,...


https://g1.globo.com/sp/sao-paulo/noticia/2026/03/29/empresario-morto-em-tentativa-de-assalto-em-sp-costumava-passear-de-moto-aos-fins-de-semana-e-voltava-de-sao-roque-com-a-esposa.ghtml

In [13]:
from newspaper import Article

def extrair_conteudo_noticia(url):
    try:
        # Configura a extração para o link fornecido
        artigo = Article(url, language='pt')
        
        # Faz o download e o processamento do HTML
        artigo.download()
        artigo.parse()
        
        # Organiza os dados em um dicionário
        dados = {
            "titulo": artigo.title,
            "texto": artigo.text,
            "data": artigo.publish_date,
            "url": url
        }
        
        return dados
    except Exception as e:
        return f"Erro ao acessar a URL: {e}"

# Teste com o link de Jequié
url_bahia = "https://ndmais.com.br/transito/motorista-causa-acidente-foge-de-abordagem-e-quase-atropela-agente-em-blumenau/"
noticia_extraida = extrair_conteudo_noticia(url_bahia)

if isinstance(noticia_extraida, dict):
    print(f"Título: {noticia_extraida['titulo']}")
    print("-" * 300)
    print(f"Texto (primeiros 300 caracteres): {noticia_extraida['texto'][:]}...")
else:
    print(noticia_extraida)

Título: VÍDEO: Motorista causa acidente, foge de abordagem e quase atropela agente em SC
------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
Texto (primeiros 300 caracteres): Um motorista foi preso na noite deste sábado (28) após fugir de uma abordagem da GMT (Guarda Municipal de Trânsito) e quase atropelar agentes em Blumenau, no Vale do Itajaí.

A ação foi registrada em vídeo e mostra o momento em que o condutor inicia a fuga, colocando os agentes em risco.

Imagens registraram o casoVídeo: Reprodução/Redes Sociais/ND Mais

Receba no WhatsApp as principais notícias de Blumenau e Vale do Itajaí

Entre no grupo

Segundo a GMT, a ocorrência começou por volta das 19h30, na rua Almirante Barroso, quando equipes foram acionadas para atender um a

In [14]:
from newspaper import Article, Config

def extrair_conteudo_noticia_forcado(url):
    try:
        # Criamos uma configuração personalizada
        user_agent = 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 Safari/537.36'
        config = Config()
        config.browser_user_agent = user_agent
        config.request_timeout = 15 # Aumentamos o tempo de espera
        
        # Passamos a config para o Article
        artigo = Article(url, config=config, language='pt')
        
        artigo.download()
        artigo.parse()
        
        # Se o texto continuar vindo curto, o site pode estar usando 'Lazy Loading' (carregamento lento)
        return {
            "titulo": artigo.title,
            "texto": artigo.text,
            "data": artigo.publish_date,
            "url": url
        }
    except Exception as e:
        return f"Erro: {e}"

# Teste novamente com o link do ND Mais
url_teste = "https://ndmais.com.br/transito/motorista-causa-acidente-foge-de-abordagem-e-quase-atropela-agente-em-blumenau/"
noticia = extrair_conteudo_noticia_forcado(url_teste)

print(f"Título: {noticia['titulo']}")
print("-" * 50)
print(f"Tamanho do texto: {len(noticia['texto'])} caracteres")
print(noticia['texto'])

Título: VÍDEO: Motorista causa acidente, foge de abordagem e quase atropela agente em SC
--------------------------------------------------
Tamanho do texto: 1911 caracteres
Um motorista foi preso na noite deste sábado (28) após fugir de uma abordagem da GMT (Guarda Municipal de Trânsito) e quase atropelar agentes em Blumenau, no Vale do Itajaí.

A ação foi registrada em vídeo e mostra o momento em que o condutor inicia a fuga, colocando os agentes em risco.

Imagens registraram o casoVídeo: Reprodução/Redes Sociais/ND Mais

Receba no WhatsApp as principais notícias de Blumenau e Vale do Itajaí

Entre no grupo

Segundo a GMT, a ocorrência começou por volta das 19h30, na rua Almirante Barroso, quando equipes foram acionadas para atender um acidente envolvendo um Honda Fit branco e um motociclista. Acidente também foi registrado por câmera:

Acidente que ocasionou a abordagem da GMTVídeo: Reprodução/Redes Sociais/ND Mais

Irregularidade motivou abordagem

Após o atendimento inicial do ac

#Teste usando Llama 70B via api por meio de Grok ou OpenRouter, que tem versoes abliterated (sem censura)

sugestao via Gemini Maromba https://gemini.google.com/app/a848d3c469eb67b5?hl=pt

Para o seu doutorado, migrar para um modelo de **70B (70 bilhões de parâmetros)** é um salto de qualidade brutal. O 70B tem uma "massa crítica" de conhecimento sociológico e linguístico que permite captar ironias, omissões sutis e enquadramentos ideológicos que o modelo de 8B (como o Dolphin) às vezes deixa passar.

Como o 70B não cabe confortavelmente na sua GPU de 16GB para rodar localmente com rapidez, a melhor forma é usar o **Groq** ou o **OpenRouter**. O Groq é absurdamente rápido (quase instantâneo) e o OpenRouter permite que você escolha modelos "Abliterated" (sem censura) se o Llama oficial começar a travar suas notícias.

### 1. Preparação (Instalação)

No seu terminal, instale a biblioteca da Groq (é a mais recomendada pela velocidade):
```bash
pip install groq
```

### 2. O Código Python (Adaptado do seu)

Vou adaptar seu código para usar a API da Groq. Você precisará de uma chave de API gratuita em [console.groq.com](https://console.groq.com/).

```python
import os
from groq import Groq

# 1. Configure sua chave de API (ou coloque direto no código, mas cuidado com segurança)
client = Groq(api_key="SUA_CHAVE_AQUI")

def classificar_70b(texto):
    # O System Prompt é onde definimos o "rigor acadêmico" do seu doutorado
    system_prompt = (
        "Você é um pesquisador sênior em Sociologia e Segurança Pública. "
        "Sua tarefa é analisar o enquadramento (framing) de notícias brasileiras. "
        "Seja analítico e identifique nuances de discurso oficial e omissões."
    )
    
    user_prompt = f"""
    Analise o texto jornalístico abaixo e classifique segundo as categorias:
    
    - [PRÓ-POLÍCIA]: Exalta a força, justifica violência ou foca no sucesso da repressão.
    - [NEUTRO]: Relato puramente factual, sem juízo de valor.
    - [CRÍTICO]: Questiona a legalidade, eficiência ou a violência da ação policial.

    Texto: {texto}
    
    Responda APENAS com a categoria entre colchetes.
    """

    # Chamada da API para o modelo Llama 3.1 70B
    chat_completion = client.chat.completions.create(
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        model="llama-3.1-70b-versatile", # Este é o modelo de 70B da Groq
        temperature=0.1,  # Baixa temperatura para manter o rigor científico e evitar "viagens"
    )

    return chat_completion.choices[0].message.content.strip()

# Exemplo de uso
noticia = "Jequié atinge marca de 30 dias sem crimes violentos após troca de comando na PM e operações da PC"
resultado = classificar_70b(noticia)
print(f"Resultado Llama-70B: {resultado}")
```

---

### 3. Por que isso é melhor para sua pesquisa de 2010?

O framework que você citou exige perceber **padrões de silenciamento** (como não dar o nome do policial). O Llama 70B consegue fazer o que chamamos de **Inferência de Intencionalidade**:

* **Dolphin 8B:** Vê "operação policial" e "sucesso" -> Classifica como Pró-polícia.
* **Llama 70B:** Percebe que o texto usa apenas fontes oficiais (SSP-BA), que não cita o nome dos agentes envolvidos em tiroteios e que usa adjetivos positivos para a PM -> Ele dá um diagnóstico muito mais preciso sobre o **discurso oficial**.

### 💡 Dica de Ouro para o Script:
Se você tiver um arquivo `.csv` ou `.xlsx` com centenas de notícias, você pode colocar esse código dentro de um `loop` e salvar os resultados em uma nova coluna. 

> **Aviso:** O Llama 70B oficial da Groq pode ocasionalmente dar o erro de "conteúdo sensível" se a notícia for extremamente violenta (descrições de tortura, etc). Se isso acontecer muito, eu te ensino a usar o **OpenRouter**, que permite acessar o **Llama 3.1 70B Instruct Abliterated** (a versão sem filtros de 70B).

**Quer que eu te ajude a montar o código para ler um arquivo de Excel e classificar tudo de uma vez?** Seria um grande avanço para a coleta de dados do seu doutorado!

In [23]:
pip install groq


   -------------------------- ------------- 2/3 [groq]
   -------------------------- ------------- 2/3 [groq]
   -------------------------- ------------- 2/3 [groq]
   -------------------------- ------------- 2/3 [groq]
   -------------------------- ------------- 2/3 [groq]
   ---------------------------------------- 3/3 [groq]

Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [27]:
import os
from groq import Groq

# 1. Substitua pela sua chave real entre as aspas
client = Groq(api_key="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") #obtive criando chave no Grok usando conta Google do Phd

def classificar_70b(texto):
    # System prompt direto e sem quebras complexas
    system_p = "Você é um pesquisador sênior em Segurança Pública. Classifique o enquadramento de notícias brasileiras de forma técnica."
    
    # Prompt de usuário simplificado para evitar erro de sintaxe
    user_p = f"Analise o texto e responda APENAS com a categoria entre colchetes: [PRÓ-POLÍCIA], [NEUTRO] ou [CRÍTICO].\n\nTexto: {texto}"

    try:
        chat_completion = client.chat.completions.create(
            messages=[
                {"role": "system", "content": system_p},
                {"role": "user", "content": user_p}
            ],
            # Use este modelo abaixo, é o substituto oficial e mais potente
            model="llama-3.3-70b-versatile", 
            temperature=0.1,
        )
        return chat_completion.choices[0].message.content.strip()
    except Exception as e:
        return f"Erro na API: {e}"

# Teste de execução
noticia = "Jequié atinge marca de 30 dias sem crimes violentos após troca de comando na PM e operações da PC"
resultado = classificar_70b(noticia)
print(f"Resultado Llama-70B: {resultado}")

Resultado Llama-70B: [PRÓ-POLÍCIA]
